# Week 4 simulations for dataset construction

In [1]:
## Install needed packages
# skip this if you have them already
#!pip install eppy esoreader tqdm shutil

In [2]:
## import needed packages

# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# EnergyPlus simulation tools
from eppy.modeleditor import IDF  # For editing EnergyPlus input files
from eppy import modeleditor
import esoreader  # For reading EnergyPlus output files

# System and file operations
import os
import sys
from itertools import product  # For creating parameter combinations
from tqdm import trange  # For progress bar
import shutil  # For file operations

def clear_output():
    """
    Clear output for both jupyter notebook and the console
    Args: None
    Returns: None
    """
    os.system('cls' if os.name == 'nt' else 'clear')  # Clear console based on OS
    if 'ipykernel' in sys.modules:
        from IPython.display import clear_output as clear
        clear()  # Clear Jupyter notebook output

In [3]:
## Define the ESO class to read the .eso file (EnergyPlus Output)
# just run this cell once

class ESO:
    def __init__(self, path):
        """Initialize ESO reader with path to .eso file"""
        self.dd, self.data = esoreader.read(path)
        
    def read_var(self, variable, frequency = "Hourly"):
        """
        Read specific variable from ESO file
        Args:
            variable: Name of the EnergyPlus output variable
            frequency: Time frequency of the data (default: Hourly)
        Returns:
            List of dictionaries containing keys and time series data
        """
        return [
            {"key": k,
             "series": self.data[self.dd.index[frequency, k, variable]]}
            for _f, k, _v in self.dd.find_variable(variable)
        ]
        
    def get_df(self, variable, frequency = "Hourly"):
        """
        Convert variable data to pandas DataFrame
        Args:
            variable: Name of the EnergyPlus output variable
            frequency: Time frequency of the data
        Returns:
            DataFrame with time series data for the variable
        """
        dic = self.read_var(variable, frequency)
        key = [each["key"] for each in dic]
        values = [each["series"] for each in dic]
        df = pd.DataFrame(values,index = key).T
        return df
    
    def total_kwh(self, variable, frequency = "Hourly"):
        """
        Calculate total energy in kWh for a given variable
        Args:
            variable: Name of the EnergyPlus output variable
            frequency: Time frequency of the data
        Returns:
            Total energy consumption in kWh
        """
        j_per_kwh = 3_600_000  # Conversion factor from Joules to kWh
        results = self.read_var(variable,frequency)
        return sum(sum(s["series"]) for s in results)/j_per_kwh

In [4]:
## Set up EnergyPlus IDD file (Input Data Dictionary)
# need to change the path to your EnergyPlus root directory

# Specify the path to EnergyPlus installation
eplus_root = r"C:\EnergyPlusV25-1-0" # Change this to your EnergyPlus root directory
iddfile = os.path.join(eplus_root,"Energy+.idd")

# Set up the IDD file for the IDF class
try:
    IDF.setiddname(iddfile)
except modeleditor.IDDAlreadySetError as e:
    pass  # Skip if IDD is already set

# Get and display current working directory
root_dir = os.getcwd()
print("Current working directory:", root_dir)

Current working directory: d:\Google Drive\My Drive\Arch 298\W4_Exercise\Part2


## Run simulation

In [5]:
## read in the base idf file
idf = IDF(os.path.join(root_dir,r"model_week4.idf"))

In [6]:
## Dictionary mapping city names to their respective weather data file paths
MAP_WEATHER = {
    "San Francisco":os.path.join(root_dir,r"..\weather_data\USA_CA_San.Francisco.Intl.AP.724940_TMY3.epw"),
    "Sacramento":os.path.join(root_dir,r"..\weather_data\USA_CA_Sacramento.Exec.AP.724830_TMY3.epw"),
    "Chicago":os.path.join(root_dir,r"..\weather_data\USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw"),
    "New York":os.path.join(root_dir,r"..\weather_data\USA_NY_New.York-J.F.Kennedy.Intl.AP.744860_TMY3.epw"),
}

MAP_WWR = {
    0.25: [2,3.75],  # Window-to-Wall Ratio 25%: [starting x coordinate, window length]
    0.40: [2,6],     # WWR 40%: [starting x coordinate, window length]
    0.60: [0.5,9],   # WWR 60%: [starting x coordinate, window length] (10*3)*0.6/2
}

## Define parameter space
Location = ["San Francisco","Sacramento","Chicago","New York"]  # Cities to simulate
Wall_Rvalue = [14.3,21.9,29.7]  # Wall thermal resistance values in ft2·°F·hr/BTU
WWR = [0.25,0.40,0.60]  # Window-to-Wall Ratio options (as decimal)
SHGC = [0.25,0.40,0.60]  # Solar Heat Gain Coefficient options for windows

# Create all possible combinations of parameters using itertools.product
keys = ["Location","Wall_Rvalue","WWR","SHGC"]
dic = [dict(zip(keys, v)) for v in product(Location,Wall_Rvalue,WWR,SHGC)]
df = pd.DataFrame(dic)  # Convert combinations to a DataFrame for easier processing
df["Wall_Rvalue"] = df["Wall_Rvalue"] * 0.1761108  # Convert Wall R-value from ft2·°F·hr/BTU to m²·K/W

In [7]:
## Modify idf function
def modify_idf(idf, location, wall_rvalue, wwr, shgc):
    """
    Modify EnergyPlus IDF object with new parameter values for the simulation
    
    Args:
        idf: EnergyPlus IDF object containing the building model
        location: City name for weather file selection
        wall_rvalue: Thermal resistance of wall (m²·K/W)
        wwr: Window-to-Wall Ratio (decimal between 0-1)
        shgc: Solar Heat Gain Coefficient (decimal between 0-1)
    
    Returns:
        Modified IDF object with updated parameters
    """
    
    # Modify wall material thermal properties
    material = idf.idfobjects["MATERIAL"]  # Get all material objects
    # Calculate new conductivity (k) from R-value and thickness
    # k = thickness/R-value (fundamental heat transfer relationship)
    new_wall_k = (1/wall_rvalue)*material[-1]["Thickness"]
    material[-1]["Conductivity"] = new_wall_k  # Update wall conductivity

    # Modify window geometry based on Window-to-Wall Ratio
    window = idf.idfobjects["WINDOW"]  # Get window objects
    new_window_x, new_window_length = MAP_WWR[wwr]  # Get position and size from mapping
    window[0]["Starting_X_Coordinate"] = new_window_x  # Set window position
    window[0]["Length"] = new_window_length  # Set window length

    # Update window material properties
    window_material = idf.idfobjects["WINDOWMATERIAL:SIMPLEGLAZINGSYSTEM"]
    window_material[0]["Solar_Heat_Gain_Coefficient"] = shgc  # Set SHGC

    # Set appropriate weather file for the location
    weather_file = MAP_WEATHER[location]  # Get weather file path from mapping
    idf.epw = weather_file  # Assign weather file to IDF
    
    return idf

In [8]:
## Run simulation
# Set working directory and initialize results DataFrame
os.chdir(root_dir)  # Change to root directory where base IDF file is located
base_path = os.path.join(root_dir)
result_df = pd.DataFrame(columns=["Total_Cooling_kwh"])  # Initialize results container

# Iterate through all parameter combinations with progress bar
for groupname in trange(len(df)):
    # Extract parameters for current simulation from DataFrame
    location = df.iloc[groupname]["Location"]        # City name for weather data
    wall_rvalue = df.iloc[groupname]["Wall_Rvalue"]  # Wall R-value in m²·K/W
    wwr = df.iloc[groupname]["WWR"]                  # Window-to-Wall Ratio
    shgc = df.iloc[groupname]["SHGC"]               # Solar Heat Gain Coefficient
    
    # Create and modify IDF for current parameter set
    idf = IDF(os.path.join(root_dir,r"model_week4.idf"))  # Load base model
    idf = modify_idf(idf,location,wall_rvalue,wwr,shgc)   # Apply parameters
    
    # Create temporary directory for simulation files
    try:
        os.mkdir(os.path.join(base_path,f"{groupname}"))   
    except:
        pass  # Directory might already exist
    
    # Run simulation and collect results
    os.chdir(os.path.join(base_path,f"{groupname}"))      # Move to simulation directory
    idf.run(expandobjects=True)                           # Run EnergyPlus simulation
    
    # Read and store results from simulation output
    eso = ESO(os.path.join(base_path,f"{groupname}\\eplusout.eso"))  # Load output
    result_df.loc[groupname] = eso.total_kwh("DistrictCooling:Facility","TimeStep")  # Get cooling energy
    
    # Clean up temporary files
    os.chdir(root_dir)                                    # Return to root directory
    clear_output()                                        # Clear progress output
    shutil.rmtree(os.path.join(base_path,f"{groupname}")) # Remove simulation directory

100%|██████████| 108/108 [03:53<00:00,  2.16s/it]


In [9]:
# Save simulation results and parameter combinations to CSV files
result_df.to_csv(os.path.join(root_dir,"week4_p2_results.csv"))  # Save cooling energy results
df.to_csv(os.path.join(root_dir,"week4_p2_all_parameters.csv"))  # Save all parameter combinations